In [13]:

import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

print(module_path)

import numpy as np
import torch
import torch.nn as nn
from hedging.envs import HedgeCallBS
from hedging.plot_utils import plot_portfolio_vs_option_price
from hedging.logit_normal import LogitNormal

from torchrl.envs import GymWrapper
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import SafeProbabilisticModule

import torch.nn as nn
from tensordict.nn import TensorDictModule, TensorDictSequential
from tensordict import TensorDict

/Users/manu13/Desktop/PHD/DeepHedging/deep_hedging_v0


In [14]:
 # --- Env Parameters (same spirit as your current train2) ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
maturity = 1.0
r = 0.05
sigma = np.array([0.15, 0.2, 0.25])
num_paths = 100
num_steps = 250
history_len = 1
transaction_cost = True
transaction_fee_rate = 1e-3

base_env = HedgeCallBS(
    S0, K, maturity, r, sigma, num_paths, num_steps,
    history_len=history_len,
    transaction_cost=transaction_cost,
    transaction_fee_rate=transaction_fee_rate
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env = GymWrapper(base_env, device=device)
env.reset(seed=0)

act_spec = env.specs["input_spec", "full_action_spec", "action"].to(device)

In [15]:
feat_dim = env.reset()["observation"].shape[-1]

hidden = 128
action_dim = 1  
inact_dim = 1   

In [16]:
class LatestState(nn.Module):
    def forward(self, obs):  # <-- obs is a plain tensor from in_keys=["observation"]
        if obs.dim() == 3:          # [B, history_len, features]
            latest = obs[:, -1, :]
        else:                       # history_len==1 or already flat
            latest = obs
        return latest               # TensorDictModule will map this to out_keys=["latest_state"]

In [17]:
class PolicyHead(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
        )
        self.mu = nn.Linear(hidden, out_dim)
        self.log_std = nn.Linear(hidden, out_dim)

    def forward(self, x):
        h = self.fc(x)
        mu = self.mu(h)
        log_std = self.log_std(h).clamp(-20, 2)
        return mu, log_std

class InactionHead(nn.Module):
    def __init__(self, in_dim, hidden, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, out_dim)  # logits
        )

    def forward(self, x):
        return self.net(x)

latest_state = TensorDictModule(
    module=LatestState(),
    in_keys=["observation"],
    out_keys=["latest_state"],
)

policy_head = TensorDictModule(
    module=PolicyHead(feat_dim, hidden, action_dim),
    in_keys=["latest_state"],
    out_keys=["mu", "log_std"],
)

inaction_head = TensorDictModule(
    module=InactionHead(feat_dim, hidden, inact_dim),
    in_keys=["latest_state"],
    out_keys=["inact_logits"],
)

In [18]:
import math

class StochasticActor(nn.Module):
    """
    Inputs are passed positionally by TensorDictModule:
      forward(mu, log_std, inact_logits) -> action, logp_action, inact, logp_inact
    """
    def forward(self, mu, log_std, inact_logits):
        std = log_std.exp()

        # Sample pre-tanh action ~ Normal(mu, std)
        eps = torch.randn_like(std)
        pre_tanh = mu + std * eps
        action = torch.tanh(pre_tanh)  # [-1, 1]

        # Log prob of Tanh-Normal via change of variables
        # log N(pre_tanh; mu, std) - sum log(1 - tanh(pre_tanh)^2)
        normal_term = -0.5 * (((pre_tanh - mu) / (std + 1e-8)) ** 2 + 2 * log_std + math.log(2 * math.pi))
        normal_log_prob = normal_term.sum(-1, keepdim=True)
        log_det_jac = torch.log(1 - action.pow(2) + 1e-8).sum(-1, keepdim=True)
        log_prob_action = normal_log_prob - log_det_jac

        # Inaction Bernoulli
        probs = torch.sigmoid(inact_logits).clamp(1e-6, 1 - 1e-6)
        bern = torch.distributions.Bernoulli(probs=probs)
        inact = bern.sample()
        log_prob_inact = bern.log_prob(inact)

        # Return tensors in the exact order of out_keys:
        # ["action_proposed", "log_prob_action", "inact", "log_prob_inact"]
        return action, log_prob_action, inact, log_prob_inact

In [19]:
stochastic_head = TensorDictModule(
    module=StochasticActor(),
    in_keys=["mu", "log_std", "inact_logits"],
    out_keys=["action_proposed", "log_prob_action", "inact", "log_prob_inact"],
)

actor = TensorDictSequential(
    latest_state,
    policy_head,
    inaction_head,
    stochastic_head,
).to(device)

In [20]:
lr = 4e-4
optim = torch.optim.Adam(actor.parameters(), lr=lr)

num_epochs = 20
num_episodes = 200
policy_only_epochs = 8
joint_training_epochs = num_epochs - policy_only_epochs
gamma = 0.999

In [21]:
def discounted_returns(rewards: torch.Tensor, gamma: float) -> torch.Tensor:
        """
        rewards: [T, B]
        returns: [T, B] with causal discounting.
        """
        T, B = rewards.shape
        t = torch.arange(T, device=rewards.device, dtype=rewards.dtype)
        powers = t[:, None] - t[None, :]
        tril = (powers >= 0).float()
        disc = (gamma ** powers.clamp_min(0)) * tril
        return disc @ rewards

In [22]:
print("=== CURRICULUM LEARNING (TorchRL) WITH TRANSACTION COSTS ===")
print(f"Phase 1: Policy-only for {policy_only_epochs} epochs")
print(f"Phase 2: Joint (policy + inaction) for {joint_training_epochs} epochs")

=== CURRICULUM LEARNING (TorchRL) WITH TRANSACTION COSTS ===
Phase 1: Policy-only for 8 epochs
Phase 2: Joint (policy + inaction) for 12 epochs


In [23]:
with set_exploration_type(ExplorationType.RANDOM):
    for epoch in range(num_epochs):
        phase = "PHASE 1: Policy Only" if epoch < policy_only_epochs else "PHASE 2: Joint"
        print(f"\nEpoch {epoch+1}/{num_epochs} — {phase}")

        for episode in range(num_episodes):
            td = env.reset(seed=epoch * 1000 + episode)
            B = td.batch_size.numel()

            # Per-episode buffers
            logp_action_list = []
            logp_inact_list  = []
            reward_list      = []
            prev_action = torch.zeros(B, action_dim, device=device)

            for t in range(num_steps):
                # Build working TD with observation
                work = TensorDict({
                    "observation": td["observation"],
                }, batch_size=[B]).to(device)

                # Actor forward
                work = actor(work)

                # Decide final action (mask with inaction only in Phase 2)
                if epoch < policy_only_epochs:
                    final_action = work["action_proposed"]
                    chosen_logp_inact = torch.zeros_like(work["log_prob_inact"])
                else:
                    mask = work["inact"].bool().expand_as(work["action_proposed"])   # <-- expand
                    final_action = torch.where(mask, prev_action, work["action_proposed"])
                    chosen_logp_inact = work["log_prob_inact"]

                # Step env
                step_td = TensorDict({
                    "action": final_action,
                }, batch_size=[B]).to(device)

                td = env.step(step_td)

                # Collect logs and rewards
                logp_action_list.append(work["log_prob_action"].squeeze(-1))
                logp_inact_list.append(chosen_logp_inact.squeeze(-1))
                reward_list.append(td["next", "reward"].squeeze(-1).detach())

                # Update prev_action for next step
                prev_action = final_action.detach()

                # Prepare for next loop
                td = td.get("next")

            logp_action = torch.stack(logp_action_list, dim=0)    
            logp_inact  = torch.stack(logp_inact_list,  dim=0)    
            rewards     = torch.stack(reward_list,      dim=0)    

            returns = discounted_returns(rewards, gamma=gamma)   
            returns = (returns - returns.mean(dim=0, keepdim=True)) / (returns.std(dim=0, keepdim=True) + 1e-8)

            logp_total = logp_action + logp_inact
            loss = (-logp_total * returns).mean()

            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(actor.parameters(), 10.0)
            optim.step()

            if (episode + 1) % 20 == 0:
                avg_r = rewards.mean().item()
                print(
                    f"  Ep {episode+1:4d}/{num_episodes} | "
                    f"Loss {loss.item():.6f} | AvgR {avg_r:.6f}"
                )


Epoch 1/20 — PHASE 1: Policy Only
  Ep   20/200 | Loss -0.660443 | AvgR -18.581236
  Ep   40/200 | Loss -1.555464 | AvgR -19.073469


KeyboardInterrupt: 

In [ ]:
class TestPolicy(nn.Module):
    def __init__(self, actor, use_inaction=False):
        super().__init__()
        self.actor = actor
        self.use_inaction = use_inaction

    def forward(self, td):
        # td is a TensorDict from the env, with key "observation"
        td = self.actor(td)

        if self.use_inaction:
            mask = td["inact"].bool().expand_as(td["action_proposed"])
            prev = td.get("prev_action", torch.zeros_like(td["action_proposed"]))
            action = torch.where(mask, prev, td["action_proposed"])
        else:
            action = td["action_proposed"]

        td.set("action", action)
        td.set("prev_action", action.detach())  # keep track if needed
        return td


In [ ]:
test_policy = TestPolicy(actor, use_inaction=True)

with set_exploration_type(ExplorationType.RANDOM):
    td_rollout = env.rollout(max_steps=num_steps, policy=test_policy)


In [ ]:
rewards = td_rollout["next","reward"]
print("Mean reward:", rewards.mean().item())

actions = td_rollout["action"]
print("First few actions:", actions[:5])


In [ ]:
plot_portfolio_vs_option_price(env._env)